In [ ]:
# Import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# Load data from Kaggle (or local path)
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Step 1: Preprocessing
# Drop useless columns
df.drop(['Name', 'Ticket', 'Cabin'], axis=1, inplace=True)

# Fill missing values
df['Age'].fillna(df['Age'].median(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# Encode categorical variables
le_sex = LabelEncoder()
df['Sex'] = le_sex.fit_transform(df['Sex'])  # male=1, female=0

le_embark = LabelEncoder()
df['Embarked'] = le_embark.fit_transform(df['Embarked'])  # S=0, C=1, Q=2

# Define X (features) and y (target)
X = df.drop('Survived', axis=1)
y = df['Survived']

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize Random Forest Classifier with ALL key parameters explained
rf = RandomForestClassifier(
    n_estimators=200,            # 👉 More trees → better stability (default=100)
    criterion='gini',            # 👉 Splitting rule: gini (faster) or entropy (more info gain)
    max_depth=10,                # 👉 Max tree depth → prevent overfitting
    min_samples_split=5,         # 👉 Min samples needed to split a node → avoid tiny splits
    min_samples_leaf=2,          # 👉 Min samples in leaf → smooth predictions
    max_features='sqrt',         # 👉 Features tried at each split → sqrt(n_features) for classification
    bootstrap=True,              # 👉 Use bootstrap sampling → enables OOB error
    oob_score=True,              # 👉 Compute OOB score during fit → internal validation
    random_state=42,             # 👉 Reproducible results
    n_jobs=-1,                   # 👉 Use all CPU cores → faster training
    verbose=1                    # 👉 Show progress (optional)
)

# Fit the model
rf.fit(X_train, y_train)

# Predict
y_pred = rf.predict(X_test)

# Evaluate
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Optional: Feature Importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)
print("\nFeature Importance:")
print(feature_importance)

# Optional: OOB Score (if bootstrap=True)
print(f"\nOOB Score: {rf.oob_score_:.4f}")

In [ ]:
# Step 1: Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Step 2: Load the data
# 'pd' is the alias for the pandas library, used for data manipulation.
# 'read_csv' is a function to read data from a CSV file into a DataFrame.
df = pd.read_csv('train.csv')

# Step 3: Data Exploration and Preprocessing
# --- Handle Missing Values ---
# The 'Age' column has missing values. We'll fill them with the median age.
# 'fillna' fills NA/NaN values with the specified value. 'median()' calculates the median.
df['Age'].fillna(df['Age'].median(), inplace=True)

# The 'Embarked' column has a few missing values. We'll fill them with the most common port.
# 'mode()[0]' gets the most frequent value.
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# --- Feature Engineering ---
# We'll create a new feature 'FamilySize' by combining 'SibSp' and 'Parch'.
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# --- Convert Categorical Features to Numerical ---
# Machine learning models need numbers, not text.
# 'Sex' is 'male' or 'female'. We'll map them to 0 and 1.
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

# 'Embarked' is a port. We'll use one-hot encoding to create new columns for each port.
# 'pd.get_dummies' converts categorical variable into dummy/indicator variables.
# 'drop_first=True' avoids multicollinearity by dropping one category.
embarked_dummies = pd.get_dummies(df['Embarked'], prefix='Embarked', drop_first=True)
df = pd.concat([df, embarked_dummies], axis=1)


# --- Select Features and Target ---
# 'features' are the columns we will use to make a prediction.
# 'target' is the column we want to predict ('Survived').
features = ['Pclass', 'Sex', 'Age', 'Fare', 'FamilySize', 'Embarked_Q', 'Embarked_S']
target = 'Survived'

X = df[features]
y = df[target]

# Step 4: Split Data into Training and Testing Sets
# 'train_test_split' splits arrays or matrices into random train and test subsets.
# 'test_size=0.2' means 20% of the data will be used for testing.
# 'random_state' ensures that the splits are the same every time we run the code, for reproducibility.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 5: Instantiate and Train the Random Forest Model with detailed parameters
# 'RandomForestClassifier' is the model class from sklearn.
# --- EXPLANATION OF PARAMETERS ---
# n_estimators=100: The number of trees in the forest. More trees can improve performance but increase training time.
# criterion='gini': The function to measure the quality of a split. 'gini' for Gini impurity, 'entropy' for information gain.
# max_depth=None: The maximum depth of a tree. If None, nodes are expanded until all leaves are pure. Can be used to prevent overfitting.
# min_samples_split=2: The minimum number of samples required to split an internal node.
# min_samples_leaf=1: The minimum number of samples required to be at a leaf node. Helps smooth the model.
# min_weight_fraction_leaf=0.0: The minimum weighted fraction of the sum total of weights required to be at a leaf node.
# max_features='sqrt': The number of features to consider when looking for the best split. 'sqrt' is a good default for classification.
# max_leaf_nodes=None: Grow trees with max_leaf_nodes in best-first fashion. Best nodes are defined as relative reduction in impurity.
# min_impurity_decrease=0.0: A node will be split if this split induces a decrease of the impurity greater than or equal to this value.
# bootstrap=True: Whether bootstrap samples are used when building trees. If False, the whole dataset is used to build each tree.
# oob_score=False: Whether to use out-of-bag samples to estimate the generalization error. We will set this to True later to show its use.
# n_jobs=None: The number of jobs to run in parallel for both fit and predict. None means 1. -1 means using all processors.
# random_state=42: Controls both the randomness of the bootstrapping of the samples and the sampling of the features. Ensures reproducibility.
# verbose=0: Controls the verbosity when fitting and predicting.
# warm_start=False: When set to True, reuse the solution of the previous call to fit and add more estimators to the ensemble.
# class_weight=None: Weights associated with classes. Useful for imbalanced datasets.
# ccp_alpha=0.0: Complexity parameter used for Minimal Cost-Complexity Pruning.
# max_samples=None: If bootstrap is True, the number of samples to draw from X to train each base estimator.
model = RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    bootstrap=True,
    oob_score=True, # Set to True to get the OOB score
    n_jobs=-1,      # Use all available CPU cores
    random_state=42
)

# The '.fit()' method trains the model on the training data.
# This is where the bootstrap sampling and tree building happens.
model.fit(X_train, y_train)

# Step 6: Make Predictions
# The '.predict()' method uses the trained forest to predict outcomes for the test set.
y_pred = model.predict(X_test)

# Step 7: Evaluate the Model
# 'accuracy_score' compares the predicted values (y_pred) with the actual values (y_test).
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")

# 'classification_report' provides precision, recall, and F1-score for each class.
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Step 8: Check Feature Importance
# 'feature_importances_' is an attribute of the trained model that shows how important each feature was.
importances = pd.DataFrame({'feature': features, 'importance': model.feature_importances_})
print("\nFeature Importances:")
print(importances.sort_values(by='importance', ascending=False))

# Step 9: Check the Out-of-Bag (OOB) Score
# The OOB score is a cross-validation estimate obtained using the out-of-bag samples.
print(f"\nOut-of-Bag (OOB) Score: {model.oob_score_:.4f}")